# Band Gap Prediction Notebook

This notebook loads the saved band-gap model checkpoint and predicts band gap from crystal-structure descriptions.

In [ ]:
# Run this once if the required packages are not installed
%pip install torch transformers sentencepiece pandas

In [ ]:
from pathlib import Path
import importlib.util

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'inference' else Path.cwd().resolve()
SCRIPT_PATH = PROJECT_ROOT / 'inference' / 'prdicted_bandgap.py'

spec = importlib.util.spec_from_file_location('prdicted_bandgap', SCRIPT_PATH)
bandgap_module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(bandgap_module)

In [ ]:
checkpoint_path = PROJECT_ROOT / 'models' / 'llmprop_band_gap_prediction_best_checkpoint.pt'
stats_csv = PROJECT_ROOT / 'data' / 'lllmprop_v.1_data_train.csv'

# Replace this with the tokenizer used during training.
# Example: tokenizer_name = 't5-small' or a local tokenizer folder path.
tokenizer_name = 't5-small'

device = bandgap_module.torch.device('cuda' if bandgap_module.torch.cuda.is_available() else 'cpu')
scaler = bandgap_module.load_scaler(stats_csv)
model = bandgap_module.build_model(checkpoint_path, device)
tokenizer = bandgap_module.load_tokenizer(tokenizer_name)

print(f'Using device: {device}')

In [ ]:
sample_text = '''Na₃Bi(P₂O₇)₂ crystallizes in the triclinic P̅1 space group. There are three inequivalent Na¹⁺ sites. In the first Na¹⁺ site, Na¹⁺ is bonded to five O²⁻ atoms to form NaO₅ square pyramids that share corners with five PO₄ tetrahedra.'''

prediction = bandgap_module.predict_single_text(
    model=model,
    tokenizer=tokenizer,
    text=sample_text,
    device=device,
    max_length=512,
    scaler=scaler,
)

print(f'Predicted band gap: {prediction:.6f} eV')

In [ ]:
# Batch prediction from a CSV file with a `description` column
input_csv = PROJECT_ROOT / 'data' / 'lllmprop_v.1_data_validation.csv'
output_csv = PROJECT_ROOT / 'inference' / 'band_gap_predictions_from_notebook.csv'

bandgap_module.predict_csv(
    model=model,
    tokenizer=tokenizer,
    input_csv=input_csv,
    output_csv=output_csv,
    text_column='description',
    device=device,
    max_length=512,
    batch_size=8,
    scaler=scaler,
)

output_csv